# CIR-ARC Phase 3.5: Model-Based Interactive ARC-AGI-3 Agent on Google Colab

This notebook demonstrates the complete **Model-Based Neuro-Symbolic Agent for ARC-AGI-3 Interactive Puzzles**.

### Agent Architecture:
1. **Epistemic BeliefState**: FactSet with provenance tracking (`FACT`, `OBSERVED`, `INFERRED`, `HYPOTHESIS`) + `UncertaintyModel`
2. **Goal Inference Engine**: Salient landmark detection, multi-factor scoring (evidence, progress, consistency, -contradiction, -cost), and hierarchical subgoal decomposition
3. **Executable World Model (Digital Twin)**: 1-step and multi-step forward simulator (`(s, a) -> s'`) with test-time replay verification
4. **Information-Gain Exploration**: Entropy-reduction action selection under uncertainty
5. **Hierarchical Planner**: Strategic Goal -> Tactical Waypoint -> A* Collision-Free Path -> Primitive Actions & Interactive Macros
6. **Failure Recovery & Model Repair**: Immediate plan invalidation, counterexample diagnosis, and oscillation breakout rollback
7. **Persistent Episodic Memory & Environment Schemas**: Long-term episode recording and schema transfer
8. **Phase 2 Perception Integration**: Neural 935K slot attention extracting continuous visual features into symbolic state

## 1. Setup & Environment Verification

In [ ]:
import os, sys
%cd /content
if not os.path.exists('/content/CIR-ARC'):
    !git clone https://github.com/Kapilraj-13/CIR-ARC.git
%cd /content/CIR-ARC
!git fetch origin master
!git reset --hard origin/master
!find . -name "*.pyc" -delete

# Ensure src is in python sys.path for the current kernel
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

!pip install -e .
!pip install scipy matplotlib pyyaml tqdm pytest
!pytest -q

## 2. Interactive Session: Solving Maze Navigation with Telemetry

In [ ]:
import sys, os
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

from cir_arc.environment.mock_engine import MockEngine
from cir_arc.solving.runtime import SolvingRuntime

# Instantiate Solving Runtime with telemetry recording
runtime = SolvingRuntime(max_actions=50, record=True)
env_maze = MockEngine("mock_maze_01")

report_maze = runtime.run_game(env_maze)
print("=" * 60)
print("=== MAZE NAVIGATION SOLVING REPORT ===")
print("=" * 60)
print(f"Game ID:          {report_maze.game_id}")
print(f"Outcome:          {report_maze.state.value} (Win: {report_maze.is_win})")
print(f"Actions Taken:    {report_maze.actions_taken}")
print(f"Elapsed Time:     {report_maze.elapsed_seconds} seconds")
print(f"Recording File:   {report_maze.recording_path}")
print("=" * 60)

## 3. Interactive Session: Multi-Stage Locksmith Puzzle (Key -> Door -> Goal)

In [ ]:
import sys, os
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

env_locksmith = MockEngine("mock_locksmith_01")
runtime_ls = SolvingRuntime(max_actions=60, record=True)

report_ls = runtime_ls.run_game(env_locksmith)
print("=" * 60)
print("=== LOCKSMITH MULTI-STAGE PUZZLE REPORT ===")
print("=" * 60)
print(f"Game ID:          {report_ls.game_id}")
print(f"Outcome:          {report_ls.state.value} (Win: {report_ls.is_win})")
print(f"Actions Taken:    {report_ls.actions_taken}")
print(f"Elapsed Time:     {report_ls.elapsed_seconds} seconds")
print("=" * 60)

## 4. Visual Inspection of Agent Progression & Grid States

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

ARC_COLORS = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
]
# Add extra color slots for interactive entities (key=11, door=8, goal=14)
full_palette = ARC_COLORS + ['#39CCCC', '#FFDC00', '#01FF70', '#85144b', '#2ECC40']
cmap = mcolors.ListedColormap(full_palette[:16])
norm = mcolors.Normalize(vmin=0, vmax=15)

# Inspect recorded action trace from locksmith run
if report_ls.recording_path and os.path.exists(report_ls.recording_path):
    with open(report_ls.recording_path, 'r') as f:
        trace = json.load(f)

    frames = trace.get('frames', [])
    print(f"Recorded {len(frames)} intermediate step frames.")

    # Display Start -> Midpoint (Key collected) -> Door Unlocked -> Goal Reached
    step_indices = [0, len(frames)//3, 2*len(frames)//3, len(frames)-1]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for i, idx in enumerate(step_indices):
        f_data = frames[min(idx, len(frames)-1)]
        g_arr = np.array(f_data.get('grid', [[]]))
        if g_arr.ndim == 3:
            g_arr = g_arr[0] + g_arr[1]
        axes[i].imshow(g_arr, cmap=cmap, norm=norm)
        axes[i].set_title(f"Step {f_data.get('step', idx)}: {f_data.get('action', 'INIT')}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

## 5. Executable World Model: Digital Twin Simulation & Replay Verification

In [ ]:
from cir_arc.world_model.executable import ExecutableWorldModel
from cir_arc.world_model.simulator import ActionSimulator
from cir_arc.world_model.replay import ReplayVerifier
from cir_arc.environment.actions import Action, ActionType

world_model = ExecutableWorldModel()
simulator = ActionSimulator(world_model=world_model)
verifier = ReplayVerifier(world_model=world_model)

# Test 1: Simulate 5-step lookahead navigation
init_frame = env_maze.reset()
plan = [Action(ActionType.ACTION2), Action(ActionType.ACTION2), Action(ActionType.ACTION4)]  # Down, Down, Right
sim_rollout = simulator.simulate_plan(init_frame, plan)

print(f"Lookahead Plan Simulation: {len(sim_rollout.predicted_frames)} frames simulated.")
print(f"Simulation Success: {sim_rollout.is_success}, Terminal State: {sim_rollout.terminal_state.value}")

# Test 2: Replay verify 1 step transition against environment
next_frame = env_maze.step(plan[0])
verification = verifier.verify_transition(init_frame, plan[0], next_frame)
print(f"Replay Verification Match: {verification.is_match} (Confidence: {verification.confidence:.2f})")

## 6. Goal Inference & Epistemic Belief State Inspection

In [ ]:
from cir_arc.belief.state import BeliefState
from cir_arc.goals.manager import GoalManager

belief = BeliefState(game_id="locksmith_demo", player_color=9)
belief.update_from_frame(env_locksmith.reset())

goal_manager = GoalManager()
comp_grid = belief.observed_composite_grid
active_goal = goal_manager.update_from_belief(belief, comp_grid)

print("=" * 60)
print("=== BELIEF STATE & GOAL INFERENCE SUMMARY ===")
print("=" * 60)
print(f"Player Location:        {belief.player_location}")
print(f"Observed Objects Count: {len(belief.observed_objects)}")
print(f"Active Strategic Goal:  {goal_manager.active_goal.goal_type.value if goal_manager.active_goal else None}")
print(f"Active Tactical Target: {goal_manager.get_active_target_coordinate()}")
print(f"Candidate Goals Count:  {len(goal_manager.candidate_goals)}")
for i, g in enumerate(goal_manager.candidate_goals):
    print(f"  [{i+1}] {g.goal_type.value} at {g.target_coordinate} (Score: {g.score:.3f}, Satisfied: {g.is_satisfied})")
print("=" * 60)

## 7. Connect Trained 935K Perception Weights into Agent Loop

In [ ]:
import torch
from cir_arc.neural.training.trainer import PerceptionModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PerceptionModel().to(device)

ckpt_path = '/content/drive/MyDrive/CIR_ARC_checkpoints/phase2/phase2_multiscale_slot_perception/best_model.pt'
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
    print("Loaded trained 935K PerceptionModel from Google Drive!")
else:
    print("Initialized PerceptionModel (Evaluate with random initialization or train in Phase 2 notebook).")

model.eval()

# Pass current game frame through neural perception model
grid_tensor = torch.tensor(comp_grid, device=device, dtype=torch.long).unsqueeze(0)
with torch.no_grad():
    neural_out = model(grid_tensor)

print(f"Neural Output Slots Shape:      {neural_out['slots'].shape} (24 discrete object slots)")
print(f"Neural Objectness Scores:       {neural_out['objectness'].shape}")
print(f"Neural Boundary Map Shape:      {neural_out['boundary_map'].shape}")
print(f"Neural Cell Objectness Shape:   {neural_out['cell_objectness'].shape}")
print("Neural perception frontend successfully wired into Phase 3 cognitive agent!")